[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/15-guided-purchase.ipynb)

## Colab setup

On Google Colab, use the setup in `tutorial/common/colab.py`. A fresh Colab runtime does not have this repository, so clone the repo into the session first. Then, before the lesson cells, run `setup_colab` from that module (import it from `tutorial.common.colab` and call it, or open the file and run `setup_colab` from there). The helper installs the tutorial packages, checks the repo out under `/content/book-agents` when it is still missing, and copies a Colab secret named `OPENAI_API_KEY`. Local Jupyter and VS Code can skip it when you already have the repo.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) before you run `setup_colab`. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

# 15. Guided purchase

The customer asks for one path: recommend a jar, put one in a cart to Ohio, check the cart, and file an order note. No card is charged.

The plan from tutorial 14 runs first. Then the recommend skill, the preference file, the catalog, `verify_cart`, and the gate. `file_order_note` waits for a person. `charge_card` stays denied even with a matching token. A 1000-jar cart is rejected before the model is asked. The page from tutorial 10 still cannot open the canary.

## Setup

Find the repo so `import tutorial` works, then load the key. `chat` calls the OpenAI Chat Completions API. The key is not printed.

In [ ]:
import sys
from pathlib import Path


def find_root():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "tutorial" / "common" / "client.py").is_file():
            return folder
    raise RuntimeError("Open this notebook inside the book-agents repository.")


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.runtime import require_key

_, model = require_key()
print("model:", model)

## Checker and gate, before the model

A bad quantity is rejected. The order note is unwritten until the token matches. The charge stays denied.

In [ ]:
import json
import re

import tutorial.common.facts
import tutorial.common.catalog
import tutorial.common.memory
import tutorial.common.skills
import tutorial.common.verify
import tutorial.common.gate
import tutorial.common.plan
import tutorial.common.trace
import tutorial.common.cost

from tutorial.common.autonomy import approval_token
from tutorial.common.cost import cost_summary, reset_cache
from tutorial.common.gate import ORDERS, gate_call, reset_orders
from tutorial.common.get_db import reset_db
from tutorial.common.harness import check, system_text
from tutorial.common.injection import show_injection_boundaries
from tutorial.common.loop import run_agent
from tutorial.common.tools import call_tool
from tutorial.common.verify import verify_cart

reset_db()
reset_orders()
reset_cache()
bad = json.loads(verify_cart({
    "sku": "HJ-CHI", "qty": 1000, "ship": True, "destination": "Ohio", "citation": "origin note",
}))
print("bad cart", bad)
check(bad["status"] == "rejected", "a 1000-jar cart is rejected")
args = {
    "customer": "Maya",
    "sku": "HJ-CHI",
    "qty": 1,
    "ship": True,
    "destination": "Ohio",
    "citation": "catalog HJ-CHI",
}
held = gate_call("file_order_note", args, set())
check(held["decision"] == "confirm_required", "the order note waits for a person")
check(not list(ORDERS.glob("*.md")), "nothing is written before approval")
allowed = gate_call("file_order_note", args, {held["approval_token"]})
check(allowed["decision"] == "allowed", "the same note is allowed when the token matches")
print(call_tool({"name": "file_order_note", "arguments": args}))
note = next(ORDERS.glob("*.md")).read_text()
print(note)
check("charged: no" in note, "the note does not charge a card")
charge = gate_call(
    "charge_card",
    {"amount_cents": 1800},
    {approval_token("charge_card", {"amount_cents": 1800})},
)
check(charge["decision"] == "denied", "a card charge stays denied")

## The page still cannot grant a tool

Same boundary as tutorial 10. The canary stays out of the output.

In [ ]:
show_injection_boundaries()

## The customer asks

Recommend, cart, verify, then the note. The first run should hold the note. The second run passes the token. Neither run charges a card.

In [ ]:
reset_orders()
question = (
    "I'm Maya. Recommend one jar I can eat, put one of those jars in a cart to ship to Ohio, "
    "check that cart, and file an order note for me. Do not charge my card."
)
first = run_agent(question, system_text(), max_steps=8, trace_id="guided-purchase")
print("ANSWER:", first["text"])
log = "\n".join(first["tool_log"])
check(bool(first["plan"]), "a plan was written before the tools")
check("load_skill" in log, "the recommend procedure was loaded")
check("verify_cart" in log, "the cart was checked")
check("charge_card" not in log or "denied" in log.lower(), "no card was charged")
tokens = re.findall(r"token=([0-9a-f]{8})", log)
print("approval tokens:", tokens)
if tokens:
    second = run_agent(
        question,
        system_text(),
        max_steps=8,
        confirmed_tokens=tokens,
        trace_id="guided-purchase-confirmed",
    )
    print("ANSWER:", second["text"])
    written = list(ORDERS.glob("*.md"))
    check(bool(written), "an order note was written after approval")
    if written:
        body = written[0].read_text()
        print(body)
        check("charged: no" in body, "the written note still does not charge a card")
print("ledger:", cost_summary(first["usage"]))
check(len(first["spans"]) >= 2, "the purchase has a trace")